# Topic Modeling with BERTopic

**Course:** [Natural Language Processing](https://ml-viz.vercel.app/courses/nlp/06-topic-modeling-bertopic)

This notebook implements the BERTopic pipeline from scratch on a synthetic toy corpus — simulated sentence embeddings via random projection, NumPy PCA for dimensionality reduction, NumPy KMeans for clustering, and class-based TF-IDF for labels. No sklearn, no API keys, no network. Just NumPy + matplotlib.

The point is to see all four stages — **embed → reduce → cluster → label** — in isolation, with code you could read end-to-end in a few minutes.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND = '#6366f1'
TEAL  = '#2dd4bf'
ROSE  = '#fb7185'
ORANGE = '#f97316'
MUTED = '#475569'
TOPIC_COLORS = [BRAND, TEAL, ROSE]

## 1. A toy support-ticket corpus

We'll hand-craft 60 short documents that fall into three obvious topics:

- **billing**: refund / invoice / charge / payment
- **shipping**: delivery / tracking / shipment / late
- **quality**: defective / broken / faulty / damaged

Each document mixes 4 topic-specific words with 3 filler words shared across topics (`the`, `is`, `my`, etc.). This is small enough to read by eye but large enough to make c-TF-IDF non-trivial.

In [ ]:
TOPIC_VOCAB = {
    'billing':  ['refund', 'invoice', 'charge',   'payment',  'card',   'bill'],
    'shipping': ['delivery','tracking','shipment', 'late',     'address','package'],
    'quality':  ['defective','broken','faulty',    'damaged',  'return', 'product'],
}
FILLER = ['the','is','my','was','i','this','order','please','need','received']
TOPIC_NAMES = list(TOPIC_VOCAB.keys())

def make_doc(rng, topic):
    body = list(rng.choice(TOPIC_VOCAB[topic], size=4, replace=True))
    filler = list(rng.choice(FILLER, size=3, replace=True))
    words = body + filler
    rng.shuffle(words)
    return ' '.join(words)

rng = np.random.default_rng(7)
true_labels = []
docs = []
for topic in TOPIC_NAMES:
    for _ in range(20):  # 20 docs per topic -> 60 total
        docs.append(make_doc(rng, topic))
        true_labels.append(topic)
true_labels = np.array(true_labels)

print(f'{len(docs)} documents, {len(set(true_labels))} ground-truth topics')
for i in [0, 1, 20, 21, 40, 41]:
    print(f'  [{true_labels[i]:<8}] {docs[i]}')

## 2. Stage 1 — Embed (simulated Sentence-BERT)

Real BERTopic uses Sentence-BERT to encode each document into a 384- or 768-dimensional vector. To keep this notebook self-contained and deterministic, we'll *simulate* that step: build a bag-of-words vector for each document, then multiply by a **frozen random projection matrix**. Random projections preserve pairwise distances (Johnson–Lindenstrauss), so semantically similar documents (same topic vocab) end up with similar projected vectors.

This is not a real sentence encoder, but it is enough to demonstrate the pipeline.

In [ ]:
# Build vocabulary across the full corpus
vocab = sorted({w for d in docs for w in d.split()})
word_to_idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f'Vocabulary size: {V}')

# Bag-of-words matrix: n_docs x V
def bow(doc):
    v = np.zeros(V)
    for w in doc.split():
        v[word_to_idx[w]] += 1.0
    return v

X_bow = np.stack([bow(d) for d in docs])
print(f'BoW shape: {X_bow.shape}')

# Frozen random-projection matrix V -> 64 (simulated embedding dim)
EMBED_DIM = 64
proj_rng = np.random.default_rng(123)
PROJ = proj_rng.normal(0, 1.0 / np.sqrt(EMBED_DIM), size=(V, EMBED_DIM))

X_emb = X_bow @ PROJ                            # shape: (n_docs, EMBED_DIM)
X_emb = X_emb / np.linalg.norm(X_emb, axis=1, keepdims=True)  # L2-normalise like SBERT
print(f'Simulated embedding shape: {X_emb.shape}')
print(f'First doc embedding norm: {np.linalg.norm(X_emb[0]):.3f}')

## 3. Stage 2 — Reduce (NumPy PCA to 2D for inspection)

Real BERTopic reduces to roughly 5 dimensions with UMAP. For this toy demo we go straight to 2D with PCA so we can scatter-plot the result. The scatter plot is a *diagnostic* — if you can already see the topics separating cleanly, the embedding is doing its job.

In [ ]:
def pca(X, n_components):
    Xc = X - X.mean(axis=0, keepdims=True)
    # SVD-based PCA: more numerically stable than eig(cov)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:n_components].T

X_2d = pca(X_emb, n_components=2)
print(f'Reduced shape: {X_2d.shape}')

fig, ax = plt.subplots(figsize=(6.5, 5))
for color, topic in zip(TOPIC_COLORS, TOPIC_NAMES):
    mask = true_labels == topic
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=topic,
               s=60, edgecolor='#0f1117', alpha=0.9)
ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
ax.set_title('2D projection of simulated sentence embeddings\n(colours = ground-truth topics)')
ax.legend(loc='best', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

Three clusters separate cleanly in 2D — exactly the picture you want before clustering. In a real BERTopic run this would be the UMAP-2D version; clustering would actually happen on a 5D UMAP projection (more separation, still low-dim enough for density to be meaningful).

## 4. Stage 3 — Cluster (NumPy KMeans, $K = 3$)

Real BERTopic uses HDBSCAN to discover $K$ automatically. For this toy demo we use KMeans with a known $K = 3$ to keep the code short. The point is the *pipeline shape*, not the choice of clusterer.

In [ ]:
def kmeans(X, K, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    # Init: pick K random data points as centroids
    idx = rng.choice(len(X), size=K, replace=False)
    centroids = X[idx].copy()
    for _ in range(n_iter):
        # Assign each point to the nearest centroid
        d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
        assignments = d2.argmin(axis=1)
        # Recompute centroids
        new_centroids = np.stack([
            X[assignments == k].mean(axis=0) if (assignments == k).any() else centroids[k]
            for k in range(K)
        ])
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids
    return assignments, centroids

# Cluster the 2D reduction (in practice you'd cluster a 5D UMAP)
assignments, centroids = kmeans(X_2d, K=3, seed=42)

# Cluster -> ground-truth agreement (greedy match)
from itertools import permutations
best_acc, best_map = 0.0, None
for perm in permutations(range(3)):
    mapped = np.array([TOPIC_NAMES[perm[c]] for c in assignments])
    acc = (mapped == true_labels).mean()
    if acc > best_acc:
        best_acc, best_map = acc, perm
print(f'Best cluster→topic mapping: {[TOPIC_NAMES[i] for i in best_map]}')
print(f'Accuracy: {best_acc:.2%}')

fig, ax = plt.subplots(figsize=(6.5, 5))
for k, c in enumerate(TOPIC_COLORS):
    mask = assignments == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=c, label=f'cluster {k}',
               s=60, edgecolor='#0f1117', alpha=0.9)
ax.scatter(centroids[:, 0], centroids[:, 1], c='white', marker='X', s=180,
           edgecolor='#0f1117', linewidths=1.5, label='centroids', zorder=5)
ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
ax.set_title(f'KMeans clusters (K=3) on the 2D reduction — accuracy {best_acc:.0%}')
ax.legend(loc='best', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

## 5. Stage 4 — Label with class-based TF-IDF

Each cluster is now a set of documents. To turn it into a *topic*, we need a label — typically the top-5 words that best characterise the cluster. **Class-based TF-IDF** treats each cluster's documents as one super-document, then ranks terms by how distinctively they appear in that cluster:

$$
\text{c-TFIDF}(t, c) \;=\; \mathrm{tf}_{t,c} \cdot \log\!\left(1 + \frac{\bar{f}}{f_t}\right)
$$

where $\mathrm{tf}_{t,c}$ is the term frequency inside cluster $c$'s super-document, $f_t$ is the total frequency of $t$ across the corpus, and $\bar{f}$ is the average number of words per cluster.

In [ ]:
def c_tf_idf(docs_by_cluster, vocab):
    """Class-based TF-IDF.

    Args:
        docs_by_cluster: dict cluster_id -> list of token lists
        vocab: sorted list of vocabulary terms

    Returns:
        score: (n_clusters, |vocab|) array of c-TF-IDF scores
    """
    n_clusters = len(docs_by_cluster)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)

    tf = np.zeros((n_clusters, V))
    for c, doc_list in docs_by_cluster.items():
        for tokens in doc_list:
            for w in tokens:
                if w in word_to_idx:
                    tf[c, word_to_idx[w]] += 1

    f_t = tf.sum(axis=0)                          # total freq of each term across corpus
    f_bar = tf.sum(axis=1).mean()                 # avg words per class
    idf_like = np.log(1.0 + f_bar / np.maximum(f_t, 1e-9))
    return tf * idf_like[None, :]

# Group docs by KMeans cluster
docs_by_cluster = {k: [] for k in range(3)}
for i, c in enumerate(assignments):
    docs_by_cluster[int(c)].append(docs[i].split())

scores = c_tf_idf(docs_by_cluster, vocab)
print(f'c-TF-IDF score matrix shape: {scores.shape}')

TOP_K = 5
for c in range(3):
    top_idx = np.argsort(scores[c])[::-1][:TOP_K]
    top_words = [vocab[i] for i in top_idx]
    print(f'Cluster {c}: {top_words}')

The top words per cluster are exactly the topic-specific vocab we planted: `refund/invoice/charge/payment/card` for billing, `delivery/tracking/shipment/late/address` for shipping, `defective/broken/faulty/damaged/return` for quality. Common filler words like `the` and `is` appear with huge term frequency in every cluster, but $\log(1 + \bar{f}/f_t)$ crushes their score to near zero — the IDF analogue doing its job at the cluster axis.

Visualise the top-5 c-TF-IDF scores per cluster:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for c, ax in enumerate(axes):
    top_idx = np.argsort(scores[c])[::-1][:TOP_K]
    words = [vocab[i] for i in top_idx]
    vals  = scores[c, top_idx]
    ax.barh(words[::-1], vals[::-1], color=TOPIC_COLORS[c], edgecolor='#0f1117')
    ax.set_title(f'Cluster {c} — top-{TOP_K} by c-TF-IDF')
    ax.set_xlabel('c-TF-IDF score')
plt.tight_layout(); plt.show()

Four-stage pipeline: done. We embedded with a random projection, reduced with PCA, clustered with KMeans, and labelled with c-TF-IDF. Swap each stage for its production counterpart — Sentence-BERT, UMAP, HDBSCAN — and you have BERTopic.

## ✏️ Your turn

### Exercise: implement `c_tf_idf_student(docs_by_cluster, vocab)`

Implement class-based TF-IDF yourself, following the formula from the lesson:

$$
\text{c-TFIDF}(t, c) = \mathrm{tf}_{t,c} \cdot \log\!\left(1 + \frac{\bar{f}}{f_t}\right)
$$

The test below builds a tiny three-cluster super-corpus and checks that the top word in cluster 0 is `refund` — i.e. that your scoring correctly down-weights filler words that appear in every cluster.

In [ ]:
def c_tf_idf_student(docs_by_cluster, vocab):
    '''Compute class-based TF-IDF.

    Args:
        docs_by_cluster: dict cluster_id -> list of token lists, e.g.
            {0: [['refund', 'the'], ['refund', 'invoice']],
             1: [['delivery', 'the'], ['tracking', 'late']],
             ...}
        vocab: sorted list of vocabulary terms.

    Returns:
        scores: (n_clusters, |vocab|) numpy array of c-TF-IDF scores.
    '''
    # TODO(you): implement the four steps
    #   1. allocate a (n_clusters, |vocab|) zero matrix `tf`
    #   2. for each cluster c, for each token list, accumulate term counts in tf[c, :]
    #   3. compute f_t = total frequency of each term across the corpus
    #            f_bar = average words per class
    #            idf_like = log(1 + f_bar / max(f_t, 1e-9))
    #   4. return tf * idf_like[None, :]
    pass

# Quick smoke run
demo_vocab = ['refund', 'invoice', 'delivery', 'tracking', 'defective', 'broken', 'the', 'is']
demo_clusters = {
    0: [['refund', 'invoice', 'the'], ['refund', 'the', 'is'], ['refund', 'invoice', 'is']],
    1: [['delivery', 'tracking', 'the'], ['delivery', 'the', 'is'], ['tracking', 'is', 'the']],
    2: [['defective', 'broken', 'the'], ['defective', 'the', 'is'], ['broken', 'defective', 'is']],
}
out = c_tf_idf_student(demo_clusters, demo_vocab)
if out is not None:
    print('Top word per cluster:')
    for c in range(3):
        top = demo_vocab[int(np.argmax(out[c]))]
        print(f'  cluster {c}: {top}')

In [ ]:
# Tests
demo_vocab = ['refund', 'invoice', 'delivery', 'tracking', 'defective', 'broken', 'the', 'is']
demo_clusters = {
    0: [['refund', 'invoice', 'the'], ['refund', 'the', 'is'], ['refund', 'invoice', 'is']],
    1: [['delivery', 'tracking', 'the'], ['delivery', 'the', 'is'], ['tracking', 'is', 'the']],
    2: [['defective', 'broken', 'the'], ['defective', 'the', 'is'], ['broken', 'defective', 'is']],
}
scores_s = c_tf_idf_student(demo_clusters, demo_vocab)
assert scores_s is not None, 'Should return an array, not None'
scores_s = np.asarray(scores_s)
assert scores_s.shape == (3, len(demo_vocab)), f'Expected (3, {len(demo_vocab)}), got {scores_s.shape}'

# Top word in cluster 0 should be `refund` (topic-specific, high tf, low cross-cluster freq).
top0 = demo_vocab[int(np.argmax(scores_s[0]))]
assert top0 == 'refund', f'Expected top word in cluster 0 to be "refund", got "{top0}"'

# Top word in cluster 1 should be `delivery` or `tracking` — both shipping-specific.
top1 = demo_vocab[int(np.argmax(scores_s[1]))]
assert top1 in {'delivery', 'tracking'}, f'Expected shipping word in cluster 1, got "{top1}"'

# Top word in cluster 2 should be `defective` or `broken` — both quality-specific.
top2 = demo_vocab[int(np.argmax(scores_s[2]))]
assert top2 in {'defective', 'broken'}, f'Expected quality word in cluster 2, got "{top2}"'

# Filler words `the` and `is` must NOT be the top word in any cluster.
for c in range(3):
    top = demo_vocab[int(np.argmax(scores_s[c]))]
    assert top not in {'the', 'is'}, f'Filler word "{top}" should not top cluster {c}'

print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def c_tf_idf_student(docs_by_cluster, vocab):
    n_clusters = len(docs_by_cluster)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)

    tf = np.zeros((n_clusters, V))
    for c, doc_list in docs_by_cluster.items():
        for tokens in doc_list:
            for w in tokens:
                if w in word_to_idx:
                    tf[c, word_to_idx[w]] += 1

    f_t   = tf.sum(axis=0)            # total term frequency across the corpus
    f_bar = tf.sum(axis=1).mean()     # average words per class
    idf_like = np.log(1.0 + f_bar / np.maximum(f_t, 1e-9))
    return tf * idf_like[None, :]
```

Three subtle points:

1. The IDF analogue is computed across **clusters**, not documents. That is the whole point of c-TF-IDF — a word like `the` that appears in every cluster gets $f_t$ ≈ corpus total, so $\bar f / f_t$ is small and $\log(1 + \dots)$ is near zero.
2. The $+1$ inside the log keeps scores non-negative even when $f_t > \bar f$ (which happens for very common words).
3. `np.maximum(f_t, 1e-9)` guards against divide-by-zero if a vocab term never appears in any document of the clusters you passed in.
</details>